# Module 4: Building the ML Pipeline Debugger — Selected PS ⭐

Walk through every component we built: models, graders, environment logic, and client.
This is the selected Problem Statement — our custom OpenEnv environment.

**Time:** ~30 min · **Difficulty:** Intermediate · **GPU:** Not required

In [ ]:
!pip install -q openenv-core torch numpy
import sys, os
sys.path.insert(0, os.path.abspath('ml-pipeline-debugger'))
print('Setup complete!')

## 1. The Models — Type-Safe Contracts

Every OpenEnv environment starts with `models.py`. These are the typed contracts between
the agent and the environment — no guessing what `obs[0][3]` means.

In [ ]:
from models import MLDebugAction, MLDebugObservation
import inspect

print('=== MLDebugAction (what agent sends) ===')
print(inspect.getsource(MLDebugAction))

print('\n=== MLDebugObservation (what agent receives) ===')
print(inspect.getsource(MLDebugObservation))

## 2. The Grader Utils — Subprocess Execution

The core of our grading system: run agent code in an isolated subprocess,
parse `loss:X.XXXXXX` from stdout, score mathematically.

In [ ]:
from server.tasks.grader_utils import run_script, parse_losses, is_decreasing, has_nan

# Test the grader pipeline on a simple training script
test_code = '''
import torch, torch.nn as nn
torch.manual_seed(42)
model = nn.Linear(5, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
X = torch.randn(50, 5); y = torch.randn(50, 1)
for i in range(10):
    loss = nn.MSELoss()(model(X), y)
    opt.zero_grad(); loss.backward(); opt.step()
    print(f"loss:{loss.item():.6f}")
'''

success, stdout, stderr = run_script(test_code, timeout=30)
losses = parse_losses(stdout)

print(f'Script ran:      {success}')
print(f'Losses parsed:   {[round(l,4) for l in losses]}')
print(f'Is decreasing:   {is_decreasing(losses)}')
print(f'Has NaN:         {has_nan(losses)}')

## 3. Task 1 — Hyperparameter Bug

Sample a bug variant, view the broken script, then grade a bad and a good fix.

In [ ]:
from server.tasks.task1_hyperparams import sample_task1, grade_task1

bug_key, broken_script, curve, description = sample_task1()

print(f'Bug injected: {bug_key}')
print(f'Description:  {description[:200]}')
print()
print('Broken script (first 15 lines):')
for i, line in enumerate(broken_script.split('\n')[:15], 1):
    print(f'  {i:3d}: {line}')

In [ ]:
# Grade the broken script (should score low)
score_bad, feedback_bad = grade_task1(broken_script)
print(f'Score (no fix): {score_bad}')
print(feedback_bad)

print()

# Grade a perfect fix (lr=0.01, 100 epochs)
perfect_fix = '''
import torch, torch.nn as nn
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(10,32), nn.ReLU(), nn.Linear(32,1))
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4)
criterion = nn.MSELoss()
X = torch.randn(200,10)
y = 3*X[:,0:1] + 2*X[:,1:2] - X[:,2:3]
for epoch in range(100):
    pred = model(X)
    loss = criterion(pred, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f"loss:{loss.item():.6f}")
'''
score_good, feedback_good = grade_task1(perfect_fix)
print(f'Score (good fix): {score_good}')
print(feedback_good)

## 4. Task 2 — NaN Data Pipeline

In [ ]:
from server.tasks.task2_nan_pipeline import sample_task2, grade_task2

bug_key, broken_script, curve, description = sample_task2()
print(f'Bug injected: {bug_key}')
print(f'Loss curve (shows NaN): {curve[:5]}')
print()

score, feedback = grade_task2(broken_script)
print(f'Score (broken): {score}')
print(feedback)

## 5. Task 3 — Silent Underfitting (the hard one)

In [ ]:
from server.tasks.task3_silent_underfit import sample_task3, grade_task3

bug_key, broken_script, curve, description = sample_task3()
print(f'Bug injected: {bug_key}')
print(f'Loss curve (looks OK but flatlines): {[round(l,4) for l in curve[:8]]}')
print()
print('Description:')
print(description[:300])

In [ ]:
# Grade broken script — scores 0.2 (runs but never escapes flatline)
score, feedback = grade_task3(broken_script)
print(f'Score (broken): {score}')
print(feedback)

## 6. The Full Environment — reset() and step()

In [ ]:
from server.ml_pipeline_debugger_environment import MlPipelineDebuggerEnvironment

env = MlPipelineDebuggerEnvironment()

print('=== 3 Episodes (one reset/step each) ===')
for i in range(3):
    obs = env.reset()
    print(f'Episode {i+1}: task={obs.task_id} difficulty={obs.difficulty} bug={obs.bug_type}')

    result = env.step(MLDebugAction(
        fixed_code=obs.broken_script,
        explanation='no fix',
        task_id=obs.task_id
    ))
    print(f'  Reward: {result.reward}  Done: {result.done}')
    print()

## 7. Determinism Check

Graders must return the same score every run. This is a hackathon disqualification criterion.

In [ ]:
bug_key, broken_script, _, _ = sample_task1()

# Grade the same script 3 times
scores = [grade_task1(broken_script)[0] for _ in range(3)]
print(f'Three runs of same script: {scores}')
print(f'Deterministic: {len(set(scores)) == 1}')

## Summary — The 3-Component Pattern

| Component | File | What it does |
|-----------|------|--------------|
| Types | `models.py` | `MLDebugAction`, `MLDebugObservation` |
| Server logic | `server/ml_pipeline_debugger_environment.py` | `reset()`, `step()` |
| Graders | `server/tasks/task*.py` | Execute fixes, score 0.0–1.0 |
| Client | `client.py` | WebSocket connection, type parsing |
| Server | `server/app.py` | FastAPI via `create_app()` |
| Container | `server/Dockerfile` | HF Spaces deployment |

**Next:** Module 5 — Training a model with GRPO to automatically debug scripts.